# Part 1: Naive RAG (Baseline)

Simple but solid baseline pipeline:
- **Chunking:** fixed size, `chunk_size=1000`, `chunk_overlap=200`
- **Embeddings:** `text-embedding-3-small`
- **Retrieval:** cosine similarity via FAISS, `top_k=5`
- **Generation:** GPT-4o-mini with retrieved chunks as context

In [1]:
import json
import os
import numpy as np
import faiss
import pypdf
from openai import OpenAI
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current working directory

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

INPUT_DIR = Path("input")
PDF_PATH  = INPUT_DIR / "ilovepdf_merged.pdf"
QA_PATH   = INPUT_DIR / "q_a.json"
OUTPUT_PATH = Path("results_naive.json")

CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 200
TOP_K         = 5
EMBED_MODEL   = "text-embedding-3-small"
GEN_MODEL     = "gpt-4o-mini"

## 1. Load PDF

In [2]:
def load_pdf(path: Path) -> list[dict]:
    """Returns list of {page_num, text} dicts."""
    reader = pypdf.PdfReader(str(path))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()
        if text:
            pages.append({"page_num": i, "text": text})
    print(f"Loaded {len(pages)} pages from {path.name}")
    return pages

pages = load_pdf(PDF_PATH)

Loaded 12 pages from ilovepdf_merged.pdf


## 2. Fixed-size chunking

In [3]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def build_chunks(pages: list[dict]) -> list[dict]:
    all_chunks = []
    for page in pages:
        for chunk in chunk_text(page["text"], CHUNK_SIZE, CHUNK_OVERLAP):
            chunk = chunk.strip()
            if chunk:
                all_chunks.append({"page_num": page["page_num"], "text": chunk})
    return all_chunks

chunks = build_chunks(pages)
print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk (page {chunks[0]['page_num']}):\n{chunks[0]['text'][:300]}...")

Total chunks: 26

Sample chunk (page 1):
5 
 
Lotte Rakhat  — это не просто производитель кондитерских изделий, это 
флагман казахстанской промышленности, соединяющий традиции и глобальные 
стандарты качества. 
В условиях усиливающейся конкуренции и интеграции в глобальную экономику 
Компания продолжит реализацию стратегии долгосрочного ус...


## 3. Embed chunks and build FAISS index

In [4]:
def get_embeddings(texts: list[str], model: str = EMBED_MODEL, batch_size: int = 100) -> np.ndarray:
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = client.embeddings.create(input=batch, model=model)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)}", end="\r")
    print()
    return np.array(all_embeddings, dtype=np.float32)

print("Embedding chunks...")
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = get_embeddings(chunk_texts)
print(f"Embeddings shape: {chunk_embeddings.shape}")

Embedding chunks...
  Embedded 26/26
Embeddings shape: (26, 1536)


In [5]:
# Normalise for cosine similarity, then use inner-product index
faiss.normalize_L2(chunk_embeddings)

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"FAISS index built with {index.ntotal} vectors (dim={dim})")

FAISS index built with 26 vectors (dim=1536)


## 4. Retrieval + Generation pipeline

In [6]:
def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    q_emb = np.array(
        client.embeddings.create(input=[query], model=EMBED_MODEL).data[0].embedding,
        dtype=np.float32
    ).reshape(1, -1)
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({"score": float(score), **chunks[idx]})
    return results


SYSTEM_PROMPT = """Ты — помощник, отвечающий на вопросы по годовому отчёту компании АО «ЛОТТЕ Рахат».
Отвечай ТОЛЬКО на основе предоставленного контекста. Если информации недостаточно — скажи об этом.
Отвечай кратко и по существу."""

def generate_answer(question: str, retrieved_chunks: list[dict]) -> str:
    context = "\n\n---\n\n".join(
        f"[Страница {c['page_num']}]\n{c['text']}" for c in retrieved_chunks
    )
    user_msg = f"Контекст:\n{context}\n\nВопрос: {question}"
    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


def run_naive_rag(question: str) -> tuple[str, list[dict]]:
    retrieved = retrieve(question)
    answer = generate_answer(question, retrieved)
    return answer, retrieved

### Quick sanity check

In [7]:
test_q = "Кто является Председателем правления АО «ЛОТТЕ Рахат»?"
test_answer, test_chunks = run_naive_rag(test_q)

print(f"Q: {test_q}")
print(f"A: {test_answer}")
print(f"\nRetrieved {len(test_chunks)} chunks from pages: {[c['page_num'] for c in test_chunks]}")

Q: Кто является Председателем правления АО «ЛОТТЕ Рахат»?
A: Председателем правления АО «ЛОТТЕ Рахат» является Ахмед Ахраров.

Retrieved 5 chunks from pages: [1, 2, 3, 1, 2]


## 5. Run on full Golden Dataset (30 Q&A)

In [8]:
with open(QA_PATH, "r", encoding="utf-8") as f:
    questions = json.load(f)

naive_results = []

for item in questions:
    print(f"[{item['id']:02d}/30] {item['question'][:60]}...")
    answer, retrieved = run_naive_rag(item["question"])
    naive_results.append({
        "id":                   item["id"],
        "question":             item["question"],
        "ground_truth":         item["ground_truth"],
        "category":             item["category"],
        "source_page":          item["source_page"],
        "predicted_answer":     answer,
        "retrieved_chunks":     [
            {"page_num": c["page_num"], "score": c["score"], "text": c["text"]}
            for c in retrieved
        ],
    })

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(naive_results, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(naive_results)} results to {OUTPUT_PATH}")

[01/30] Кто является Председателем правления АО «ЛОТТЕ Рахат»?...
[02/30] В каком году было начато производство кондитерских изделий в...
[03/30] Какая доля акций перешла к LOTTE Confectionery Co. LTD в 201...
[04/30] Какое количество наименований продукции насчитывается в порт...
[05/30] Какой объем экспорта в денежном выражении (USD) был в 2024 г...
[06/30] Какая страна является основным экспортером кондитерской прод...
[07/30] Перечислите 4 пункта стратегии долгосрочного развития на 202...
[08/30] Как изменилось название компании в 2021 году?...
[09/30] Сколько твердо-бытовых отходов было передано на полигон в 20...
[10/30] Какой объем сточных вод составил за отчетный период 2024 год...
[11/30] Каковы фактические выбросы газообразных и жидких веществ для...
[12/30] Превышают ли фактические выбросы ЛОС установленный лимит на ...
[13/30] Чему равна сумма фактических выбросов (Всего) по всем трем п...
[14/30] Насколько лимит твердых выбросов на Фабрике больше фактическ...
[15/30] Сравн

## 6. Quick results preview

In [9]:
print(f"{'ID':<4} {'Cat':<10} {'Ground Truth':<45} {'Predicted':<45}")
print("-" * 110)
for r in naive_results:
    gt  = r["ground_truth"][:43]
    ans = r["predicted_answer"][:43]
    print(f"{r['id']:<4} {r['category']:<10} {gt:<45} {ans:<45}")

ID   Cat        Ground Truth                                  Predicted                                    
--------------------------------------------------------------------------------------------------------------
1    Simple     Ахмед Ахраров                                 Председателем правления АО «ЛОТТЕ Рахат» яв  
2    Simple     1942 год                                      В предоставленном контексте нет информации   
3    Simple     ~ 76%                                         В 2013 году LOTTE Confectionery Co. LTD при  
4    Simple     Более 450 наименований                        В портфеле АО «ЛОТТЕ Рахат» насчитывается б  
5    Simple     156,7 млн. $                                  Объем экспорта кондитерских изделий из Каза  
6    Simple     Российская Федерация                          Основным экспортером кондитерской продукции  
7    Simple     1. Модернизация производственных мощностей;   1. Модернизация производственных мощностей.  
8    Simple     Официальн

In [10]:
# Hit Rate @ 5: did the correct source_page appear in any of the top-5 retrieved chunks?
hits = 0
reciprocal_ranks = []

for r in naive_results:
    retrieved_pages = [c["page_num"] for c in r["retrieved_chunks"]]
    source = r["source_page"]
    if source in retrieved_pages:
        hits += 1
        rank = retrieved_pages.index(source) + 1
        reciprocal_ranks.append(1 / rank)
    else:
        reciprocal_ranks.append(0)

hit_rate = hits / len(naive_results)
mrr = np.mean(reciprocal_ranks)

print(f"Hit Rate @ {TOP_K}: {hit_rate:.3f}  ({hits}/{len(naive_results)})")
print(f"MRR:               {mrr:.3f}")

# Per-category breakdown
from collections import defaultdict
cat_hits = defaultdict(list)
for r, rr in zip(naive_results, reciprocal_ranks):
    cat_hits[r["category"]].append(rr > 0)

print("\nHit Rate by category:")
for cat, values in cat_hits.items():
    print(f"  {cat:<12}: {sum(values)}/{len(values)} = {sum(values)/len(values):.3f}")

Hit Rate @ 5: 0.967  (29/30)
MRR:               0.786

Hit Rate by category:
  Simple      : 9/10 = 0.900
  Table       : 10/10 = 1.000
  Synthesis   : 10/10 = 1.000
